In [ ]:
#imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if not openrouter_api_key:
    raise ValueError("OPENROUTER_API_KEY is not set in the environment variables.")
else:
    print(f"OPENROUTER_API_KEY exists and begins {openrouter_api_key[:8]}...")

In [ ]:
#Initialize OpenRouter client
openrouter_url="https://openrouter.ai/api/v1"
openrouter=OpenAI(api_key=openrouter_api_key,base_url=openrouter_url)
MODEL='gemini-2.5-flash-lite'

In [ ]:
system_prompt="""
You are AnimeCraft, an AI assistant designed to help aspiring
manga artists plan and improve their manga panels.

Help the user with:
- camera angles
- panel composition
- character positioning
- perspective
- lighting
- shadows
- background elements
- dynamic movements
- speed lines
- anatomy
- hatching and shading techniques
- movement
- facial expressions
- visual storytelling

Give practical and actionable advice that an artist can
actually use while drawing.

When appropriate, structure your suggestions clearly.And finally give a beginner friendly response, use less tokens and give more clarity and end with conclusion."""

In [ ]:
def craft_response(message,history):
    history=[{'role':h['role'],'content':h['content']} for h in history]
    messages=[{'role':'system','content':system_prompt}]+history+[{'role':'user','content':message}]
    response=openrouter.chat.completions.create(model=MODEL,messages=messages,max_completion_tokens=500)
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=craft_response,title="AnimeCraft").launch()

In [ ]:
#Streaming version of the response function
def craft_response(message,history):
    history=[{'role':h['role'],'content':h['content']} for h in history]
    messages=[{'role':'system','content':system_prompt}]+history+[{'role':'user','content':message}]
    stream=openrouter.chat.completions.create(model=MODEL,messages=messages,max_completion_tokens=500,stream=True)
    response=""
    for chunk in stream:
        response+=chunk.choices[0].delta.content or ''
        yield response
    

In [ ]:
gr.ChatInterface(fn=craft_response,title="Anime Craft").launch(auth=("Drona", "AnimeCraft"))

Crafting the responses with several conditional system instructions

In [ ]:
def craft_response(message,history):
    history=[{'role':h['role'],"content":h['content']} for h in history]
    relevant_system_prompt=system_prompt
    if "action" in message.lower():
        relevant_system_prompt += """
    For action scenes, focus on dynamic poses, body movement,
    perspective, impact, motion lines, and panel composition.
    Suggest camera angles that make the action feel energetic
    and visually clear.
    """

    if any(word in message.lower() for word in
       ["perspective", "vanishing point", "foreshortening"]):
        relevant_system_prompt += """
    When discussing perspective, explain vanishing points,
    horizon lines, depth, scale, and foreshortening.
    Explain how objects and characters should change in size
    and shape as their distance from the viewer changes.
    Give practical advice that can be applied while drawing.
    """

    if "composition" in message.lower():
        relevant_system_prompt += """
    When discussing composition, explain how to arrange
    characters, objects, foreground, midground, background,
    and negative space. Help establish a clear focal point
    and guide the reader's eye through the panel.
    """

    if any(word in message.lower() for word in
       ["facial expression", "facial expressions", "expression"]):
        relevant_system_prompt += """
    When discussing facial expressions, focus on the eyes,
    eyebrows, mouth, head angle, and subtle changes in facial
    features. Explain how these elements communicate emotions
    clearly in a manga panel.
    """

    if any(word in message.lower() for word in
       ["background", "environment", "setting"]):
        relevant_system_prompt += """
    When discussing backgrounds, explain how the environment
    can establish location, depth, atmosphere, and mood.
    Suggest important environmental details while avoiding
    unnecessary details that could distract from the characters.
    """

    if any(word in message.lower() for word in
       ["camera angle", "camera", "shot"]):
        relevant_system_prompt += """
    When discussing camera angles, recommend an appropriate
    viewpoint such as eye-level, low-angle, high-angle,
    bird's-eye view, or Dutch angle based on the intended
    emotion and storytelling purpose. Explain why the chosen
    viewpoint works for the scene.
    """

    if any(word in message.lower() for word in
       ["lighting", "light", "illumination"]):
        relevant_system_prompt += """
    When discussing lighting, explain the direction, intensity,
    and quality of light and how it affects the mood, depth,
    and readability of the manga panel. Consider how light
    interacts with the characters and environment.
    """

    if any(word in message.lower() for word in
       ["shadow", "shadows", "shading"]):
        relevant_system_prompt += """
    When discussing shadows or shading, explain where shadows
    should fall based on the light source, how they can create
    depth and volume, and how hard or soft shadows can influence
    the mood of the scene. Consider cast shadows, form shadows,
    and strong manga-style contrast when appropriate.
    """
    messages=[{'role':'system','content':'relevant_system_prompt'}]+history+[{'role':'user','content':message}]
    stream=openrouter.chat.completions.create(model=MODEL,messages=messages,stream=True,max_tokens=2000)
    response=""
    for chunk in stream:
        response+=chunk.choices[0].delta.content or ""
        yield response
    

In [ ]:
gr.ChatInterface(fn=craft_response,title='''AnimeCraft V1 Manga & Visual Storytelling Assistant''').launch()

🎬 AnimeCraft V1 test questions
Feature	Test question
Action	“I want to draw a character jumping from one rooftop to another. How should I make the action feel dynamic?”
Perspective	“How can I draw a character's hand reaching toward the viewer using foreshortening?”
Composition	“How should I compose a manga panel where two characters are arguing in a small room?”
Facial expressions	“How can I draw a convincing angry expression without making it look exaggerated?”
Background	“How should I design a busy street background without making it distract from the main character?”
Camera angle	“Which camera angle would make a character look powerful and intimidating?”
Lighting	“How should I light a character standing beside a window at sunset?”
Shadows	“Where should the shadows fall if the main light source is above and to the left of the character?”
Now test combinations

These are more interesting because multiple conditions should trigger.

1. Action + camera angle

“I'm drawing a sword fight. What camera angle and composition would make the attack feel powerful and dynamic?”

2. Perspective + action

“A character is punching toward the viewer. How should I use perspective and foreshortening?”

3. Lighting + shadows

“A character is standing under a streetlight at night. How should I handle the lighting and shadows?”

4. Composition + background

“I'm drawing two characters talking in a train station. How should I compose the panel and design the background?”

5. Facial expression + camera angle

“I want a close-up of a character realizing they've been betrayed. What camera angle and facial expression should I use?”

And finally, one "normal" question

Test something that shouldn't trigger any special condition:

“What are some general tips for improving my manga drawings?”

That lets you verify that your base system prompt still works even when none of the conditional instructions are added.

Q:

I'm drawing a manga scene set on the streets of Tokyo, Japan. A middle-school boy notices a TV playing a volleyball match. A team from his prefecture is competing at the national tournament for the first time. He becomes fascinated after seeing one of the players perform an incredible spike. From that moment, he decides that he wants to play volleyball, with the dream of eventually becoming a spiker himself.

I want this scene to capture the feeling of a young character discovering his passion for the first time, similar to the emotional feeling of a sports manga. Give me a sequence of camera angles, perspective choices, and panel compositions for this scene. Explain why each shot works and how the shots should transition from the boy casually watching the TV to the moment he becomes inspired.